# 2. Build and test the models

**Question:** can a model predict which orders arrive late, and how many days delivery will take, better than simple rules?

The honest way to answer is to give simple rules every chance first. Every model has to beat them.

**Evaluation rules used throughout**
- **Time-based split.** Oldest 70% of orders train the models, the next 15% (validation) tunes them, the newest 15% (test)
  is touched once at the end. Orders are always judged on a period they have not seen.
- **Shipped orders only** (cancelled orders removed, see notebook 1).
- **Metrics:** ROC-AUC, PR-AUC, *precision at 80% recall* (of the orders we flag to catch 80% of the late ones, how many are
  truly late) and *lift in the top 20%*. F1 is not used: flagging every order already scores 0.71.
- **Tie-aware.** A rule gives many orders the same score. Ties are broken randomly before PR-AUC, precision-at-recall
  and lift are computed, otherwise rules are unfairly penalised (ROC-AUC is unaffected).

Run this notebook after notebook 1. It saves `models/models.joblib`, which notebook 3 and the app use.

In [4]:
import joblib, numpy as np, pandas as pd, matplotlib.pyplot as plt
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
import core
from core import CAT, NUM, FEATURES, MODE, TARGET, TARGET_DAYS, SEED

df = core.load_dataco()
data = core.shipped_only(df)
train, val, test = core.time_split(data)
pd.DataFrame({"orders": [len(d) for d in (train, val, test)],
              "from": [d[core.DATE].min().date() for d in (train, val, test)],
              "to": [d[core.DATE].max().date() for d in (train, val, test)],
              "late rate": [round(d[TARGET].mean(), 3) for d in (train, val, test)]}, index=["train", "validation", "test"])

,orders,from,to,late rate
train,120935,2015-01-01,2017-01-07,0.574
validation,25915,2017-01-07,2017-06-13,0.566
test,25915,2017-06-13,2018-01-31,0.576


## 1. Part A: will the order be late? The leaderboard

From the dumbest to the best:
- **constant**: everyone gets the training late rate (ROC-AUC 0.5 by definition)
- **mode only**: each shipping mode gets its training late rate
- **5-row lookup table**: shipping mode, with Same Day split at the noon cut-off (what notebook 1 suggests)
- **logistic regression** and **random forest** on all features, and again **without** shipping mode and promised days
  (what do the other columns add on their own?)
- **LightGBM**, tuned by a small random search on the validation split (a few minutes)

In [5]:
base = train[TARGET].mean()
mode_rate = train.groupby(MODE)[TARGET].mean()
lookup = train.assign(key=core.rule_key(train)).groupby("key")[TARGET].mean()

probs = {"constant": np.full(len(test), base),
         "mode only": test[MODE].map(mode_rate).fillna(base).to_numpy(),
         "5-row lookup table": pd.Series(core.rule_key(test)).map(lookup).to_numpy()}

for label, feats in {"all features": FEATURES,
                     "without mode / promise": [c for c in FEATURES if c not in (MODE, core.PROMISE)]}.items():
    cat = [c for c in feats if c in CAT]; num = [c for c in feats if c in NUM]
    for name, clf, scale in [("logistic regression", LogisticRegression(max_iter=1000, class_weight="balanced"), True),
                             ("random forest", RandomForestClassifier(150, min_samples_leaf=20, class_weight="balanced", n_jobs=-1, random_state=SEED), False)]:
        m = Pipeline([("prep", core.make_preprocessor(cat, num, scale)), ("clf", clf)]).fit(train[feats], train[TARGET])
        probs[f"{name} ({label})"] = m.predict_proba(test[feats])[:, 1]

late, params, val_auc = core.fit_late_model(train, val)            # LightGBM, tuned on validation
probs["LightGBM (tuned)"] = late.predict_proba(test[FEATURES])[:, 1]
print("best LightGBM settings:", params, "| validation ROC-AUC:", round(val_auc, 4))

board = pd.DataFrame({k: core.score_row(test[TARGET], p) for k, p in probs.items()}).T.sort_values("roc_auc", ascending=False)
board

best LightGBM settings: {'num_leaves': 31, 'n_estimators': 200, 'min_child_samples': 20, 'learning_rate': 0.05} | validation ROC-AUC: 0.771


,roc_auc,pr_auc,precision@80%recall,lift@top20%
LightGBM (tuned),0.779,0.860,0.679,1.71
5-row lookup table,0.777,0.860,0.677,1.70
random forest (all features),0.777,0.857,0.679,1.68
logistic regression (all features),0.752,0.842,0.653,1.65
mode only,0.746,0.840,0.653,1.65
logistic regression (without mode / promise),0.522,0.591,0.582,1.05
random forest (without mode / promise),0.521,0.588,0.582,1.04
constant,0.500,0.576,0.577,0.99


**How to read the leaderboard**
- If a model's ROC-AUC is about the same as the **5-row lookup table**, it has learned nothing a business rule does not know.
  That is the central finding to check for here.
- If *without mode / promise* is close to 0.5, the other columns (region, product, payment type) carry almost no signal.
- *precision@80%recall*: a random flag gives the base late rate; higher is better.

## 2. Does removing cancelled orders matter?

Train and test LightGBM on *all* orders and on *shipped only*, and compare each with mode-only. If the gap over mode-only shrinks
when cancelled orders go, part of the old edge was predicting cancellation (through payment type), not lateness.

In [6]:
rows = []
for pop, d in {"all orders": df, "shipped only": data}.items():
    tr, va, te = core.time_split(d)
    mrate = tr.groupby(MODE)[TARGET].mean()
    rows.append({"population": pop, "model": "mode only", **core.score_row(te[TARGET], te[MODE].map(mrate).fillna(tr[TARGET].mean()))})
    m = late if pop == "shipped only" else core.fit_late_model(tr, va)[0]
    rows.append({"population": pop, "model": "LightGBM", **core.score_row(te[TARGET], m.predict_proba(te[FEATURES])[:, 1])})
pd.DataFrame(rows).set_index(["population", "model"])

roc_auc  pr_auc  precision@80%recall  lift@top20%
population   model                                                       
all orders   mode only    0.731   0.797                0.623         1.64
             LightGBM     0.774   0.839                0.659         1.70
shipped only mode only    0.746   0.840                0.653         1.65
             LightGBM     0.779   0.860                0.679         1.71

## 3. What does LightGBM actually use?

Two experiments. First: remove the three order-time features (hour, weekday, month). Second: compare the honest time split
with a random split.

In [7]:
no_time = [c for c in NUM if c not in ("order_hour", "order_weekday", "order_month")]
m = Pipeline([("prep", core.make_preprocessor(CAT, no_time)), ("clf", LGBMClassifier(**params, **core.LATE_FIXED))]).fit(train[CAT + no_time], train[TARGET])
auc_no_time = roc_auc_score(test[TARGET], m.predict_proba(test[CAT + no_time])[:, 1])
auc_full = roc_auc_score(test[TARGET], probs["LightGBM (tuned)"])
print(f"test ROC-AUC with order-time features: {auc_full:.3f}   without: {auc_no_time:.3f}")
print("Order hour carries real signal: the Same Day noon cut-off.")

test ROC-AUC with order-time features: 0.779   without: 0.745
Order hour carries real signal: the Same Day noon cut-off.


In [8]:
def fit_auc(tr, te):
    m = Pipeline([("prep", core.make_preprocessor()), ("clf", LGBMClassifier(**params, **core.LATE_FIXED))]).fit(tr[FEATURES], tr[TARGET])
    return round(roc_auc_score(te[TARGET], m.predict_proba(te[FEATURES])[:, 1]), 3)

n = len(data); idx = np.random.RandomState(SEED).permutation(n)
rand_train, rand_test = data.iloc[idx[:int(.70 * n)]], data.iloc[idx[int(.85 * n):]]
print(f"share of rows whose order has several items: {data['Order Id'].duplicated(keep=False).mean():.2f}")
pd.DataFrame({"test ROC-AUC": {"time split (honest)": fit_auc(train, test), "random split": fit_auc(rand_train, rand_test)}})

share of rows whose order has several items: 0.89


,test ROC-AUC
time split (honest),0.779
random split,0.799


Items of one order share the same shipping mode, order time and delivery time. A **random** split puts some
items of an order in training and the others in test, so the model recognises its test rows. That inflates the score.
That is why every result in this project uses the time split.

## 4. Part B: how many days will delivery take, and how sure are we?

A point guess is not enough: the customer needs a window. Target: real delivery days. We compare simple baselines, then
build an **80% window** three ways. Coverage should be near 80% (80% of real deliveries fall inside) with the window as
narrow as possible.

In [9]:
eta = core.eta_pipeline().fit(train[FEATURES], train[TARGET_DAYS])
mode_mean = train.groupby(MODE)[TARGET_DAYS].mean()
pd.DataFrame({"predict the promise": core.regression_row(test[TARGET_DAYS], test[core.PROMISE]),
              "mean real days per mode": core.regression_row(test[TARGET_DAYS], test[MODE].map(mode_mean)),
              "LightGBM": core.regression_row(test[TARGET_DAYS], eta.predict(test[FEATURES]))}).T

,mae,rmse,r2
predict the promise,1.294,1.600,0.019
mean real days per mode,0.982,1.267,0.385
LightGBM,0.973,1.265,0.387


In [10]:
ALPHA = 0.2; G = test[MODE]
res = {}
# 1) per-mode quantiles of the training data
emp = core.empirical_intervals(train, MODE, ALPHA)
lo, hi = core.apply_intervals(emp, G, (train[TARGET_DAYS].quantile(ALPHA / 2), train[TARGET_DAYS].quantile(1 - ALPHA / 2)))
res["per-mode quantiles"] = core.interval_row(test[TARGET_DAYS], lo, hi, G)
# 2) LightGBM quantile regression
qlo = core.eta_pipeline(objective="quantile", alpha=ALPHA / 2).fit(train[FEATURES], train[TARGET_DAYS])
qhi = core.eta_pipeline(objective="quantile", alpha=1 - ALPHA / 2).fit(train[FEATURES], train[TARGET_DAYS])
a, b = qlo.predict(test[FEATURES]), qhi.predict(test[FEATURES])
res["LightGBM quantile regression"] = core.interval_row(test[TARGET_DAYS], np.minimum(a, b), np.maximum(a, b), G)
# 3) split conformal: half-width per mode = the 80% quantile of |real - predicted| on the validation split
hw = core.conformal_halfwidths(val[TARGET_DAYS], eta.predict(val[FEATURES]), val[MODE], ALPHA)
lo, hi = core.conformal_interval(eta.predict(test[FEATURES]), G, hw)
res["split conformal (per mode)"] = core.interval_row(test[TARGET_DAYS], lo, hi, G)

intervals = pd.DataFrame(res).T
print("conformal half-widths (days):", {k: round(v, 2) for k, v in hw.items()})
intervals[[c for c in intervals.columns if c in ("coverage", "mean_width") or c.startswith(("cov_", "width_"))]]

conformal half-widths (days): {'First Class': 0.04, 'Same Day': 0.07, 'Second Class': 2.0, 'Standard Class': 2.0, '__all__': 1.98}


,coverage,mean_width,cov_First Class,width_First Class,cov_Same Day,width_Same Day,cov_Second Class,width_Second Class,cov_Standard Class,width_Standard Class
per-mode quantiles,1.000,3.230,1.000,0.000,1.000,1.000,1.000,4.000,1.000,4.000
LightGBM quantile regression,0.899,3.233,0.978,0.178,0.713,1.016,0.688,3.884,0.966,3.998
split conformal (per mode),0.790,3.191,0.731,0.076,0.839,0.118,0.800,4.001,0.796,3.997


**How to read this.** Real days are whole numbers and about uniform from 2 to 6 inside Second and Standard Class, so the
window there is wide (several days) however clever the method. First Class always takes exactly 2 days, so its window has width 0.
If three methods give the same width, the width is a property of the **data**, not of the method. The app uses the
conformal windows.

## 5. Save the models

Notebook 3 and the app load this file.

In [11]:
core.MODELS_FILE.parent.mkdir(exist_ok=True)
joblib.dump({"late": late, "eta": eta, "halfwidths": hw, "late_params": params}, core.MODELS_FILE)
print("saved", core.MODELS_FILE)

saved D:\ml\delivery_reliability\models\models.joblib


## Findings (fill in from your own output)

1. Best classifier and the 5-row lookup table (ROC-AUC, precision@80%recall): 
2. What the other columns add (the "without mode / promise" rows): 
3. All orders vs shipped only: 
4. Order-time features (with / without): 
5. Time split vs random split: 
6. Delivery window: coverage and width by mode: 